# Gemma 4 E2B × TinyCeNN — Integrated Memory V2.1

This notebook fixes the loader interception bug seen in the previous balanced run.

**What changed**
- the benchmark now patches the real `Gemma4ForCausalLM.from_pretrained` class method;
- `model.language_model.*` checkpoint keys are remapped to `model.*`;
- core text-weight loading is validated before training;
- the untouched native Gemma-4 cached/full path must pass a sanity gate before TinyCeNN is judged;
- the default profile remains **`balanced`**.

TinyCeNN V1 safely targets only independent Gemma-4 global-attention layers:
- conservative: layer `4`
- expanded: layers `4,9`

Shared-KV producer/consumer layers remain native.


In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN detected.')
else:
    print('No HF_TOKEN secret found; public download will be attempted.')

REPO = Path(tempfile.mkdtemp(prefix='gemma4-e2b-cenn-v21-')) / 'TinyCeNN-LM'
subprocess.run(['git','clone','--quiet','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run(['git','fetch','origin','main'], cwd=REPO, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=REPO, check=True)

subprocess.run([
    sys.executable,'-m','pip','install','-q',
    'transformers==5.17.0','huggingface_hub>=0.36.2','datasets>=3,<6',
    'accelerate','pytest','pandas','matplotlib'
], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'], check=True)

os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO),str(REPO/'src')])
sys.path[:0] = [str(REPO),str(REPO/'src')]

import torch
from transformers import AutoConfig

MODEL_ID = 'google/gemma-4-E2B'
cfg = AutoConfig.from_pretrained(MODEL_ID, token=HF_TOKEN).get_text_config(decoder=True)
FULL = [i for i,t in enumerate(cfg.layer_types) if t == 'full_attention']
FIRST_SHARED = cfg.num_hidden_layers - cfg.num_kv_shared_layers
NONSHARED_FULL = [i for i in FULL if i < FIRST_SHARED]
SHARED_KV_PRODUCER = max(NONSHARED_FULL)
SAFE_REPLACEABLE = [i for i in NONSHARED_FULL if i != SHARED_KV_PRODUCER]
SOURCE = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()

print('Source:', SOURCE)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('GPU VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), 'GiB')
print('Model type:', cfg.model_type)
print('Layers:', cfg.num_hidden_layers)
print('Full attention:', FULL)
print('First KV-shared layer:', FIRST_SHARED)
print('Shared-KV producer:', SHARED_KV_PRODUCER)
print('Safely replaceable:', SAFE_REPLACEABLE)

assert cfg.model_type == 'gemma4_text'
assert FULL == [4,9,14,19,24,29,34]
assert cfg.num_kv_shared_layers == 20
assert SAFE_REPLACEABLE == [4,9]


In [ ]:
PROFILE = 'balanced' # @param ['smoke','balanced','extended']
SAVE_TO_DRIVE = True # @param {type:'boolean'}

PROFILES = {
    'smoke': dict(
        train_contexts='96,128', test_contexts='96,128,256',
        block_size=16, features=32,
        train_documents=4, validation_documents=2, test_documents=2, warm_documents=2,
        warm_steps=2, joint_steps=4, eval_every=2,
        timing_documents=1, timing_repeats=1, decode_tokens=8, loss_chunk=2,
    ),
    'balanced': dict(
        train_contexts='128,256', test_contexts='128,256,512,1024',
        block_size=32, features=64,
        train_documents=24, validation_documents=6, test_documents=8, warm_documents=4,
        warm_steps=30, joint_steps=60, eval_every=15,
        timing_documents=1, timing_repeats=1, decode_tokens=24, loss_chunk=4,
    ),
    'extended': dict(
        train_contexts='128,256,512', test_contexts='128,256,512,1024,2048',
        block_size=32, features=96,
        train_documents=48, validation_documents=10, test_documents=16, warm_documents=8,
        warm_steps=60, joint_steps=120, eval_every=20,
        timing_documents=2, timing_repeats=2, decode_tokens=32, loss_chunk=4,
    ),
}

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime for Gemma 4 E2B.')

VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
if PROFILE == 'balanced' and VRAM_GB < 30:
    print('⚠️ Balanced may require an A100-class runtime. If CUDA OOM occurs, switch only for debugging to smoke.')

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/TinyCeNN/gemma4-e2b-integrated-v2-1')
else:
    BASE = Path('/content/gemma4-e2b-integrated-v2-1')

BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = BASE / run_id
LOG = BASE / (run_id + '.log')
RUN = dict(PROFILES[PROFILE], seed=2041)

print('Profile:', PROFILE)
print(json.dumps(RUN, indent=2))
print('Results:', OUT)


## Preflight

This preflight checks three different things:
1. TinyCeNN/Gemma-4 adapter tests.
2. Multimodal → text checkpoint-remapping test.
3. The exact V2.1 loader-interception mechanism that failed in the previous notebook.

The third check is important: it verifies that `v1.main()` will see the patched **real class method**.


In [ ]:
subprocess.run(['git','fetch','origin','main'], cwd=REPO, check=True)
subprocess.run(['git','reset','--hard','origin/main'], cwd=REPO, check=True)
SOURCE = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Testing source:', SOURCE)

env = dict(os.environ, CUDA_VISIBLE_DEVICES='', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
tests = ['tests/test_gemma4_checkpoint.py','tests/test_gemma4_integrated_memory.py','tests/test_gemma4_v21_loader_intercept.py']
r = subprocess.run(
    [sys.executable,'-m','pytest','-q',*tests],
    cwd=REPO, env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(r.stdout)
if r.returncode:
    raise RuntimeError(f'Gemma 4 V2.1 regression preflight failed: {r.returncode}')

BENCHMARK = REPO / 'scripts/benchmark_gemma4_e2b_integrated_memory_v2.py'

# Verify the exact lazy-import bug is fixed without downloading the 10.2 GB checkpoint.
probe_code = r'''
import sys
sys.path[:0] = [r"''' + str(REPO) + r'''", r"''' + str(REPO/'src') + r'''"]
import scripts.benchmark_gemma4_e2b_integrated_memory_v2 as b
from transformers import Gemma4ForCausalLM
assert getattr(Gemma4ForCausalLM.from_pretrained, "__func__", None) is b._mapped_from_pretrained.__func__
print("REAL_CLASSMETHOD_INTERCEPT_OK")
'''
probe = subprocess.run(
    [sys.executable,'-c',probe_code],
    cwd=REPO, env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(probe.stdout)
if probe.returncode or 'REAL_CLASSMETHOD_INTERCEPT_OK' not in probe.stdout:
    raise RuntimeError('Gemma 4 V2.1 loader interception preflight failed')

help_probe = subprocess.run(
    [sys.executable,str(BENCHMARK),'--help'],
    cwd=REPO, env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(help_probe.stdout.splitlines()[0] if help_probe.stdout else '')
if help_probe.returncode:
    raise RuntimeError('Gemma 4 V2.1 benchmark entrypoint failed')

print('✅ V2.1 preflight passed — real classmethod interception is active')


## Train + evaluate — balanced profile

The first important runtime line after the checkpoint load must be:

`gemma4_text_loader_v2_1: {"core_missing_keys": [], "unexpected_text_keys": [], ...}`

If the real text weights are not loaded, the benchmark stops before training.


In [ ]:
cmd = [sys.executable,'-u',str(BENCHMARK),'--base-model',MODEL_ID,'--output-dir',str(OUT)]
for k,v in RUN.items():
    cmd += ['--' + k.replace('_','-'), str(v)]

print('Running:', BENCHMARK.name)
print(' '.join(cmd))

try:
    with LOG.open('w') as log:
        with subprocess.Popen(
            cmd, cwd=REPO,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=os.environ.copy()
        ) as p:
            saw_loader_ok = False
            for line in p.stdout:
                print(line,end='',flush=True)
                log.write(line); log.flush()
                if 'gemma4_text_loader_v2_1:' in line and '"core_missing_keys": []' in line and '"unexpected_text_keys": []' in line:
                    saw_loader_ok = True
            status = p.wait()

    if not saw_loader_ok:
        raise RuntimeError(
            'V2.1 did not confirm a clean Gemma-4 text-weight load. '
            'Do not use results from this run.'
        )
    if status:
        raise RuntimeError(f'Run failed: {status}; inspect {LOG}')
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG, OUT/'console.log')
        print('Archive:', shutil.make_archive(str(OUT)+'-results','zip',root_dir=OUT))


## Results and baseline sanity check

Besides the benchmark's cache gate, this cell checks that the original model does not have random-model-like held-out NLL.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

s = pd.read_csv(OUT/'integrated_summary.csv')
selected = json.loads((OUT/'selection.json').read_text())['selected']

original_rows = s[s.candidate == 'original']
if original_rows.empty:
    raise RuntimeError('No original Gemma-4 baseline rows were produced.')

max_original_nll = float(original_rows.test_nll.max())
print('Original max held-out NLL:', max_original_nll)
if max_original_nll >= 10.0:
    raise RuntimeError(
        f'Original Gemma-4 NLL={max_original_nll:.3f} is suspiciously close to a random model. '
        'Do not interpret this run.'
    )

cols = [c for c in [
    'candidate','context','test_nll','test_perplexity','ppl_ratio','adapted_ppl_ratio',
    'total_cache_ratio','prefill_speedup','decode_speedup','remaining_full_attention_layers',
    'cached_logits_nmse','teacher_cached_logits_nmse','candidate_top1_mismatches',
    'teacher_top1_mismatches','allowed_top1_mismatches',
    'cache_equivalence_top1_floor','cache_equivalence_passed','selected_on_validation'
] if c in s.columns]

print('Locked validation selection:', selected)
display(s[cols].sort_values(['candidate','context']).reset_index(drop=True))

x = s[s.candidate == selected].sort_values('context')
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(x.context,x.ppl_ratio,marker='o',label='PPL ratio')
ax.plot(x.context,x.total_cache_ratio,marker='s',label='cache ratio')
ax.axhline(1,linestyle='--')
ax.set_xscale('log',base=2)
ax.set_xlabel('Context')
ax.legend()
ax.set_title(selected)
plt.show()


## Sample user prompts — original Gemma 4 vs selected TinyCeNN

This uses the same validated checkpoint remapping for the original model and deterministic greedy decoding for both models.


In [ ]:
import gc
from transformers import AutoTokenizer
from transformers.cache_utils import DynamicCache
from tinycenn_lm.gemma4_checkpoint import load_gemma4_text_causal
from tinycenn_lm.gemma4_integrated_memory import (
    restore_student, new_cache, inference_mode, native_dtype, text_config
)

manifest = json.loads((OUT/'manifest.json').read_text())
report = json.loads((OUT/'integrated_report.json').read_text())
SELECTED = json.loads((OUT/'selection.json').read_text())['selected']
record = next(r for r in report['candidates'] if r['candidate'] == SELECTED)
checkpoint = OUT / record['checkpoint']

device = torch.device('cuda')
dtype = native_dtype(device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=manifest['model_revision'], token=HF_TOKEN
)

original, load_info = load_gemma4_text_causal(
    MODEL_ID,
    revision=manifest['model_revision'],
    dtype=dtype,
    attn_implementation='sdpa',
    token=HF_TOKEN,
    device=device,
)
assert load_info['core_missing_keys'] == []
assert load_info['unexpected_text_keys'] == []

payload = torch.load(checkpoint,map_location='cpu',weights_only=True)
cenn = restore_student(original,payload).to(device).eval()

@torch.no_grad()
def native_generate(model, ids, max_new_tokens=64):
    cache = DynamicCache(config=text_config(model))
    logits = model(
        input_ids=ids,past_key_values=cache,use_cache=True,logits_to_keep=1
    ).logits[:,-1]
    out=[]
    eos = tokenizer.eos_token_id
    eos = {eos} if isinstance(eos,int) else set(eos or [])
    for _ in range(max_new_tokens):
        tok = logits.argmax(-1,keepdim=True)
        out.append(tok)
        if int(tok.item()) in eos:
            break
        logits = model(
            input_ids=tok,past_key_values=cache,use_cache=True,logits_to_keep=1
        ).logits[:,-1]
    return torch.cat(out,1)

@torch.no_grad()
def cenn_generate(model, ids, max_new_tokens=64):
    cache = new_cache(model)
    out=[]
    eos = tokenizer.eos_token_id
    eos = {eos} if isinstance(eos,int) else set(eos or [])
    with inference_mode(model,str(dtype).replace('torch.','')):
        logits = model(
            input_ids=ids,past_key_values=cache,use_cache=True,logits_to_keep=1
        ).logits[:,-1]
        for _ in range(max_new_tokens):
            tok = logits.argmax(-1,keepdim=True)
            out.append(tok)
            if int(tok.item()) in eos:
                break
            logits = model(
                input_ids=tok,past_key_values=cache,use_cache=True,logits_to_keep=1
            ).logits[:,-1]
    return torch.cat(out,1)

TEST_PROMPTS = [
    'Explain in simple words why the sky is blue.',
    'Calculate 18 times 7 and explain the calculation briefly.',
    'Write a short Python function that returns whether a number is prime.',
    'A train travels 120 km in 90 minutes. What is its average speed in km/h?',
    'What are three practical differences between RAM and SSD storage?',
    'Summarize in one sentence why attention caches use memory during autoregressive decoding.',
]

rows=[]
for i,prompt in enumerate(TEST_PROMPTS,1):
    ids = tokenizer(prompt,return_tensors='pt').input_ids.to(device)
    a = native_generate(original,ids,64)
    b = cenn_generate(cenn,ids,64)

    common=min(a.shape[1],b.shape[1])
    agreement=float((a[:,:common]==b[:,:common]).float().mean()) if common else 0.0

    a_text=tokenizer.decode(a[0],skip_special_tokens=True).strip()
    b_text=tokenizer.decode(b[0],skip_special_tokens=True).strip()

    print('\n'+'='*100)
    print(f'TEST {i}: {prompt}')
    print('\nORIGINAL GEMMA 4:\n',a_text)
    print('\nTinyCeNN:\n',b_text)
    print(f'\nGenerated-token agreement: {agreement:.2%}')

    rows.append({
        'prompt':prompt,
        'original':a_text,
        'cenn':b_text,
        'token_agreement':agreement,
        'original_tokens':a.shape[1],
        'cenn_tokens':b.shape[1],
    })

chat_results=pd.DataFrame(rows)
display(chat_results[['prompt','token_agreement','original_tokens','cenn_tokens']])
chat_results.to_csv(OUT/'chat_comparison.csv',index=False)
(OUT/'chat_comparison.json').write_text(json.dumps(rows,indent=2,ensure_ascii=False))

print('Mean generated-token agreement:', f"{chat_results.token_agreement.mean():.2%}")
print('Saved:', OUT/'chat_comparison.csv')
